# 项目-航空AI助手

现在，我们将汇集我们所学到的知识，为航空公司打造人工智能客户支持助理

In [ ]:
# 导入

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

In [ ]:
# 初始化

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

DB = "prices.db"

In [ ]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, friendly and courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer after checking all available tools, say so.
Use the provided functions to answer questions about ticket prices and flight status before responding.
"""

In [ ]:
# get_ticket_price 的 JSON 函数规范
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

# get_flight_status 的 JSON 函数规范
flight_status_function = {
    "name": "get_flight_status",
    "description": "Check the status of a booked flight by flight number and date.",
    "parameters": {
        "type": "object",
        "properties": {
            "flight_number": {
                "type": "string",
                "description": "The flight number to check status for",
            },
            "date": {
                "type": "string",
                "description": "The date of the flight (YYYY-MM-DD)",
            },
        },
        "required": ["flight_number", "date"],
        "additionalProperties": False
    }
}

# 更新模型使用工具
tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": flight_status_function}
]
tools

In [ ]:
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"

In [ ]:

flights_record_status = [
    {"LN123": {"date": "2026-03-05", "status": "On Time"}},
    {"PR456": {"date": "2026-03-12", "status": "Delayed"}},
    {"TK789": {"date": "2026-03-18", "status": "Cancelled"}},
    {"BL321": {"date": "2026-03-22", "status": "On Time"}}
]
flights_record_status

In [ ]:
def get_flight_status(flight_number: str) -> dict:
    """Fetches the status of a flight given its flight number from flights_record_status."""
    for flight in flights_record_status:
        if flight_number in flight:
            flight_info = flight[flight_number]
            return {
                'flight_number': flight_number,
                'status': flight_info['status']
            }
    # 如果没有找到，则返回未知状态
    return {
        'flight_number': flight_number,
        'status': 'Unknown'
    }

In [ ]:
# 将 get_flight_status 集成到 Flight Assistant 逻辑中

def flight_assistant(user_input: str):
    """
    Handles user requests for booking flights or checking flight status.
    """
    if "status" in user_input.lower():
        # 从用户输入中提取航班号（占位符逻辑）
        flight_number = "LN123"  # Example, replace with extraction logic
        return get_flight_status(flight_number)
    else:
        # 从用户输入中提取目的地（占位符逻辑）
        destination = "Paris"  # Example, replace with extraction logic
        return get_ticket_price(destination)


In [ ]:

def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat).launch()

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
        elif tool_call.function.name == "get_flight_status":
            arguments = json.loads(tool_call.function.arguments)
            flight_number = arguments.get('flight_number')
            status_details = get_flight_status(flight_number)
            responses.append({
                "role": "tool",
                "content": json.dumps(status_details),
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
gr.ChatInterface(fn=chat).launch()

## 更多关于 Gradio 实际用途的信息：

1. Gradio 基于我们对 UI 的 Python 描述构建了一个前端 Svelte 应用程序
2. Gradio 启动一个基于 Starlette Web 框架构建的服务器，监听为该 React 应用程序提供服务的空闲端口
3. Gradio 为我们的回调创建后端路由，例如 chat()，它调用我们的函数

当然，当 Gradio 生成前端应用程序时，它会确保“提交”按钮调用正确的后端路由。

就是这样！

它很简单，而且结果让人感觉很神奇。

#我们走多式联运吧！

我们可以使用 GPT-4o 背后的图像生成模型 DALL-E-3 来制作一些图像

让我们把它放在一个名为艺术家的函数中。

### 价格提醒：每次生成图像的成本约为 4 美分 - 不要对图像着迷！

In [ ]:
# Some 导入 for handling images

import base64
from io import BytesIO
from PIL import Image

In [ ]:
def artist(city):
    image_response = openai.images.generate(
            model="dall-e-3",
            prompt=f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style",
            size="1024x1024",
            n=1,
            response_format="b64_json",
        )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

In [ ]:
image = artist("New York City")
display(image)

In [ ]:
def talker(message):
    response = openai.audio.speech.create(
      model="gpt-4o-mini-tts",
      voice="onyx",    # Also, try replacing onyx with alloy or coral
      input=message
    )
    return response.content

## 让我们把这个带回家：

1. 具有图像和音频生成功能的多模态人工智能助手
2. 工具调用与数据库查找
3. 迈向 Agentic 工作流程的一步

In [ ]:
def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    cities = []
    image = None

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    voice = talker(reply)

    if cities:
        image = artist(cities[0])
    
    return history, voice, image


In [ ]:
def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            cities.append(city)
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses, cities

## Gradio UI 的 3 种类型

`gr.Interface` 用于标准、简单的 UI

`gr.ChatInterface` 用于标准 ChatBot UI

`gr.Blocks` 用于自定义 UI，您可以在其中控制组件和回调

In [ ]:
# 回调（以及上面的 chat() 函数）

def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# 用户界面定义

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500)
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# 将事件连接到回调

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

ui.launch(inbrowser=True, auth=("ed", "bananas"))

# 练习和商业应用

添加更多工具 - 或许可以模拟实际预订航班。一名学生已经做到了这一点，并在社区贡献文件夹中提供了他们的示例。

下一步：将其应用到您的业务中。使用可以为您的工作执行活动的工具制作一个多模式人工智能助手。客户支持助理？新员工入职助理？这么多的可能性！另请参阅单独笔记本中的第 2 周周末练习。

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="宽度：150px；高度：150px；垂直对齐：中间；">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">我对你有一个特殊要求</h2>
            <span style="color:#090;">
                我的编辑告诉我，学生在 Udemy 上对这门课程进行评分会产生巨大的影响 - 这是 Udemy 决定是否向其他人展示该课程的主要方式之一。如果您能花一点时间评价一下，我将非常感激！无论如何，如果我可以随时提供帮助，请随时通过 ed@edwarddonner.com 与我联系。
            </span>
        </td>
    </tr>
</表>